# Drivers & Sankey Analysis — SQL Database

This notebook loads all data directly from the SQLite database.

**Data sources:**
- `database/database.db` → `processed` table (articles + driver columns)
- `database/database.db` → `Threats_Clean` table (threat decoded labels)

In [ ]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display

DB_PATH = Path('../database/database.db')
print(f'DB exists: {DB_PATH.exists()}')

## Step 1: Load data from SQLite database

In [ ]:
conn = sqlite3.connect(DB_PATH)

df_articles = pd.read_sql('SELECT * FROM processed', conn)
df_threats  = pd.read_sql('SELECT * FROM Threats_Clean', conn)

conn.close()

print(f'Articles: {df_articles.shape}')
print(f'Threats rows: {df_threats.shape}')
df_articles[['ArticleID', 'Direct_driver_clean', 'Indirect_driver_clean']].head(3)

In [ ]:
df_threats_agg = (
    df_threats.groupby('ArticleID', as_index=False)
    .agg({'Threat1_decoded': lambda x: '; '.join(sorted(set(x.dropna().astype(str))))})
)
df_threats_agg['ArticleID'] = df_threats_agg['ArticleID'].astype(str)

print(f'Aggregated threats: {df_threats_agg.shape}')
df_threats_agg.head(3)

## Step 3: Merge articles with threats

In [ ]:
df_articles['ArticleID'] = df_articles['ArticleID'].astype(str)

df = df_articles.merge(df_threats_agg, on='ArticleID', how='left')

print(f'Merged dataset: {df.shape}')
print(f'Non-null Direct_driver_clean:   {df["Direct_driver_clean"].notna().sum()}')
print(f'Non-null Indirect_driver_clean: {df["Indirect_driver_clean"].notna().sum()}')
print(f'Non-null Threat1_decoded:       {df["Threat1_decoded"].notna().sum()}')
df[['ArticleID', 'Direct_driver_clean', 'Indirect_driver_clean', 'Threat1_decoded']].head(6)

## Step 4: Driver Frequency Bar Charts

In [ ]:
def explode_drivers(series, sep=', '):
    return (
        series
        .dropna()
        .str.split(sep)
        .explode()
        .str.strip()
        .loc[lambda s: (s != '') & (s.str.lower() != 'unknown')]
    )

direct_counts = explode_drivers(df['Direct_driver_clean']).value_counts().reset_index()
indirect_counts = explode_drivers(df['Indirect_driver_clean']).value_counts().reset_index()
direct_counts.columns = ['driver', 'count']
indirect_counts.columns = ['driver', 'count']

fig_direct = px.bar(
    direct_counts.sort_values('count'),
    x='count', y='driver', orientation='h',
    title='Individual Direct Driver Frequency',
    labels={'count': 'Number of articles', 'driver': ''},
    color='count',
    color_continuous_scale='Teal',
    text='count',
)
fig_direct.update_traces(textposition='outside')
fig_direct.update_layout(coloraxis_showscale=False, height=350, margin=dict(l=200))
display(fig_direct)

fig_indirect = px.bar(
    indirect_counts.sort_values('count'),
    x='count', y='driver', orientation='h',
    title='Individual Indirect Driver Frequency',
    labels={'count': 'Number of articles', 'driver': ''},
    color='count',
    color_continuous_scale='Oryel',
    text='count',
)
fig_indirect.update_traces(textposition='outside')
fig_indirect.update_layout(coloraxis_showscale=False, height=380, margin=dict(l=230))
display(fig_indirect)

## Step 5: Co-occurrence Heatmap — Indirect × Direct Driver

In [ ]:
rows_expanded = []
for _, row in df.iterrows():
    directs   = [d.strip() for d in str(row['Direct_driver_clean']).split(',')
                 if d.strip() and d.strip().lower() != 'unknown']
    indirects = [i.strip() for i in str(row['Indirect_driver_clean']).split(',')
                 if i.strip() and i.strip().lower() != 'unknown']
    for ind in indirects:
        for dir_ in directs:
            rows_expanded.append({'indirect': ind, 'direct': dir_})

df_pairs = pd.DataFrame(rows_expanded)
cooccurrence = (
    df_pairs
    .groupby(['indirect', 'direct'])
    .size()
    .reset_index(name='count')
    .pivot(index='indirect', columns='direct', values='count')
    .fillna(0)
    .astype(int)
)

fig_heat = px.imshow(
    cooccurrence,
    title='Co-occurrence: Indirect Driver × Direct Driver',
    labels=dict(x='Direct Driver', y='Indirect Driver', color='Articles'),
    color_continuous_scale='Blues',
    text_auto=True,
    aspect='auto',
)
fig_heat.update_layout(height=450, margin=dict(l=220, b=160))
fig_heat.update_xaxes(tickangle=-30)
display(fig_heat)

## Step 6: Sankey Diagram — Indirect Driver → Direct Driver → Threat

In [ ]:
sankey_rows = []
for _, row in df.iterrows():
    indirects = [i.strip() for i in str(row['Indirect_driver_clean']).split(',')
                 if i.strip() and i.strip().lower() != 'unknown']
    directs   = [d.strip() for d in str(row['Direct_driver_clean']).split(',')
                 if d.strip() and d.strip().lower() != 'unknown']
    threats   = [t.strip() for t in str(row['Threat1_decoded']).split(';')
                 if t.strip() and t.strip().lower() not in ('unknown', 'nan')]

    for ind in indirects:
        for dir_ in directs:
            sankey_rows.append({'source': f'IND: {ind}', 'target': f'DIR: {dir_}'})
    for dir_ in directs:
        for thr in threats:
            sankey_rows.append({'source': f'DIR: {dir_}', 'target': f'THR: {thr}'})

df_sankey = (
    pd.DataFrame(sankey_rows)
    .groupby(['source', 'target'])
    .size()
    .reset_index(name='value')
)

all_nodes = list(dict.fromkeys(df_sankey['source'].tolist() + df_sankey['target'].tolist()))
node_idx = {n: i for i, n in enumerate(all_nodes)}

def node_color(label):
    if label.startswith('IND:'): return 'rgba(255, 160, 86, 0.85)'
    if label.startswith('DIR:'): return 'rgba(70, 145, 210, 0.85)'
    return 'rgba(100, 180, 120, 0.85)'

node_colors = [node_color(n) for n in all_nodes]
node_labels = [n.split(': ', 1)[1] for n in all_nodes]

fig_sankey = go.Figure(go.Sankey(
    arrangement='snap',
    node=dict(
        pad=18,
        thickness=20,
        line=dict(color='white', width=0.5),
        label=node_labels,
        color=node_colors,
    ),
    link=dict(
        source=[node_idx[s] for s in df_sankey['source']],
        target=[node_idx[t] for t in df_sankey['target']],
        value=df_sankey['value'].tolist(),
        color='rgba(180, 180, 180, 0.3)',
    ),
))

fig_sankey.update_layout(
    title_text='Causal Chain: Indirect Driver → Direct Driver → Threat',
    title_font_size=16,
    height=650,
    font=dict(size=12),
    annotations=[
        dict(x=0.01, y=1.04, xref='paper', yref='paper',
             text='<b>Indirect Drivers</b>', showarrow=False,
             font=dict(color='rgba(255,140,40,1)', size=13)),
        dict(x=0.50, y=1.04, xref='paper', yref='paper',
             text='<b>Direct Drivers</b>', showarrow=False,
             font=dict(color='rgba(50,120,200,1)', size=13)),
        dict(x=0.99, y=1.04, xref='paper', yref='paper',
             text='<b>Threats</b>', showarrow=False,
             font=dict(color='rgba(60,150,80,1)', size=13)),
    ],
)
display(fig_sankey)